# Episode-level k-fold dynamics study (plan `docs/plans/dynamics-kfold-plan.md`, frozen before any fold ran)

**Question.** Does the pilot's world model beat persistence on DROID-100 episodes it has not seen, and how often would a random ten-episode split say so? The parity run (`docs/research-notes/2026-09-30-dynamics-parity.md`) found one checkpoint passing the 2 % gate on the test split and failing it on the validation split, ten episodes each.

**What it does.** Five folds over all 100 episodes, split by episode before windowing (fold = rank by SHA-256 of the episode id, modulo 5; 20 held-out episodes per fold). For each held-out fold it trains one model on 70 of the other 80 episodes (10 of them select the checkpoint), scores the 20 held-out episodes against persistence, and pools all 100. **One regime per job** (`regimes`): `pilot` (`mask_mode='subset'`, `prediction_mode='full'`) or `v2` (`'single_choice'`, `'base'`); everything else is pilot v2's configuration. Before the folds, an **anchor unit** trains on the pilot's own split and its best validation score must reproduce the stored number (v2: the probe's `dynamics_gate.json`; pilot: the `best_metrics` in its checkpoint) within a tolerance fixed in the plan; otherwise the report says `ANCHOR_FAILED`, writes the tables, and makes no classification.

**Result classes (frozen in the plan).** On the pooled relative improvement over persistence `R` with a 95 % episode-cluster bootstrap interval and the gate margin 0.02: `BEATS` (lower bound above 0.02), `WORSE_THAN_PERSISTENCE` (upper bound below 0), `FAILS` (upper bound below 0.02), `INCONCLUSIVE` (otherwise). Also reported, with no decision attached: the share of random ten-episode splits that would pass the gate, `R` per fold and per horizon step, the share of episodes on which the model beats persistence, and the secondary numbers (final-step weights, the other prediction mode). Per-episode errors are saved so that later questions need no rerun.

**Cost and storage.** About 55 minutes on an L4 per regime (six trainings of about 8 minutes). Checkpoints stay on the VM's local disk and are deleted after each unit (their SHA-256 values are recorded); only small files go to Drive. Resumable at unit granularity with `RESUME_RUN_ID`.

**Limits (plan §7).** One training seed per fold; 70 training episodes instead of the pilots' 80 (the anchor unit uses 80); DROID-100 only; normalised units. The k-fold models are diagnostics: nothing here is evidence for or against H2, and B2 does not reuse them. The pilot's ten test episodes are held out once and used for training in the other folds; the anchor unit does not read them.


In [ ]:
# ================================================================
# 0. Fetch the tested package and pin its commit
# ================================================================
import os
import subprocess
import sys

REPO_URL = 'https://github.com/romusking/para_001'
REPO_REF = 'main'  # branch, tag or commit SHA. The job worker pins the commit here.
REPO_DIR = '/content/para_001'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--all'], check=False)


def resolve_ref(ref):
    for candidate in (f'origin/{ref}', ref):
        result = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--verify', '--quiet', candidate + '^{commit}'], capture_output=True, text=True)
        if result.returncode == 0:
            return result.stdout.strip()
    raise RuntimeError(f'Cannot resolve {ref!r} in {REPO_URL}.')


subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', '--detach', resolve_ref(REPO_REF)], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
REPO_DIRTY = bool(subprocess.check_output(['git', '-C', REPO_DIR, 'status', '--porcelain'], text=True).strip())
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

try:
    import adjointrwm
    import adjointrwm.models
    import adjointrwm.training  # noqa: F401
    from adjointrwm.eval import kfold_assignment  # noqa: F401
except ImportError as error:
    raise RuntimeError(f'{REPO_REF!r} ({REPO_COMMIT[:12]}) predates the k-fold helpers in adjointrwm.eval.dynamics_parity. Set REPO_REF to a commit that contains them.') from error
print('adjointrwm', adjointrwm.__version__, 'at', REPO_COMMIT, '(uncommitted changes!)' if REPO_DIRTY else '')

In [ ]:
# ================================================================
# 1. Runtime, frozen configuration, run directory
# ================================================================
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

import dataclasses
import datetime
import json
import platform
import shutil
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from adjointrwm.io import atomic_copy, atomic_write_json, atomic_write_text, sha256_file, sha256_json

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
DEVICE = torch.device('cuda')
GPU_NAME = torch.cuda.get_device_name(0)
if 'L4' not in GPU_NAME:
    raise RuntimeError(f'This study is scheduled for an L4; this runtime has {GPU_NAME}.')
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

REGIMES = {'pilot': {'mask_mode': 'subset', 'prediction_mode': 'full'}, 'v2': {'mask_mode': 'single_choice', 'prediction_mode': 'base'}}


@dataclass(frozen=True)
class KFoldConfig:
    regimes: tuple = ('v2',)   # one job per regime: ('pilot',) or ('v2',)
    # Sources: the feature cache and the stored anchors.
    v1_run_id: str = 'droid100_adjoint_20260929T070629Z'
    v1_checkpoint_sha256: str = '4b0f07177221ae63467bd0ce7f8ae97545e9fc92a27b3a0731551648709fcc3d'
    v2_run_id: str = 'droid100_adjoint_v2_20260930T165409Z'
    v2_checkpoint_sha256: str = '66c80f464a442da0ba9e3ee424aef692a1be4b76eff860fe1d8640c08555882b'
    # Design (plan section 2).
    k: int = 5
    inner_validation_episodes: int = 10
    frame_stride: int = 2
    context_len: int = 8
    horizon: int = 4
    window_stride: int = 2
    # Model and training: pilot v2's configuration.
    d_model: int = 512
    transformer_layers: int = 6
    transformer_heads: int = 8
    transformer_ff: int = 2048
    dropout: float = 0.10
    num_refinement_candidates: int = 4
    candidate_costs: tuple = (1.0, 1.0, 1.5, 2.0)
    rate_beta: float = 0.002
    seed_base: int = 20260928
    steps: int = 1500
    batch_size: int = 64
    learning_rate: float = 3.0e-4
    weight_decay: float = 0.05
    warmup_steps: int = 100
    gradient_clip: float = 1.0
    eval_interval: int = 100
    checkpoint_interval: int = 250
    eval_batch_size: int = 128
    # Decision rule and outputs (plan sections 3 and 4).
    margin: float = 0.02
    bootstrap_resamples: int = 10000
    bootstrap_seed: int = 0
    subset_size: int = 10
    subset_draws: int = 10000
    # Policy values fixed in the plan, not measured: how closely the anchor unit must reproduce a stored number.
    anchor_atol_v2: float = 0.01
    anchor_atol_pilot: float = 0.02


CFG = KFoldConfig()
assert CFG.regimes and set(CFG.regimes) <= set(REGIMES), f'unknown regime in {CFG.regimes}'
CONFIG_DICT = asdict(CFG)
CONFIG_HASH = sha256_json(CONFIG_DICT)

RESUME_RUN_ID = None  # e.g. 'dynamics_kfold_v2_20261001T090000Z'; skips the units that finished

DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production')
SESSION_ID = datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
    RUN_DIR = DRIVE_ROOT / 'runs' / RUN_ID
    stored = json.loads((RUN_DIR / 'config' / 'run_config.json').read_text())
    if stored['config_sha256'] != CONFIG_HASH:
        raise RuntimeError('This notebook config differs from the paused run. Resume with the identical config.')
    if stored['repo_commit'] != REPO_COMMIT:
        raise RuntimeError(f"The run used package commit {stored['repo_commit']}; set REPO_REF to it.")
else:
    RUN_ID = 'dynamics_kfold_' + '-'.join(CFG.regimes) + '_' + SESSION_ID
    RUN_DIR = DRIVE_ROOT / 'runs' / RUN_ID
    if RUN_DIR.exists():
        raise RuntimeError(f'{RUN_DIR} already exists; run directories are immutable.')
PATHS = {name: RUN_DIR / name for name in ('config', 'artifacts', 'logs', 'reports')}
UNIT_DIR = PATHS['artifacts'] / 'units'
LOCAL = Path('/content/adjoint_rwm_work') / RUN_ID
for path in [LOCAL, UNIT_DIR, *PATHS.values()]:
    path.mkdir(parents=True, exist_ok=True)

if not RESUME_RUN_ID:
    atomic_write_json(PATHS['config'] / 'run_config.json', {
        'run_id': RUN_ID, 'config': CONFIG_DICT, 'config_sha256': CONFIG_HASH, 'repo_commit': REPO_COMMIT, 'repo_dirty': REPO_DIRTY, 'gpu': GPU_NAME,
        'notebook': 'notebooks/02-diagnostics/dynamics_kfold.ipynb', 'plan': 'docs/plans/dynamics-kfold-plan.md',
        'kfold_uses_all_100_episodes': True, 'anchor_reads_pilot_test_episodes': False,
        'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__,
    })
print('Run:', RUN_ID, '| regimes:', CFG.regimes, '| GPU:', GPU_NAME, '| config', CONFIG_HASH[:12], '| resumed' if RESUME_RUN_ID else '| new')

In [ ]:
# ================================================================
# 2. Feature cache of the pilot v2 probe run, folds, the pilot's own split (for the anchor unit only)
# ================================================================
from adjointrwm.data import WindowDataset, WindowSpec, assignment_from_manifest, episode_split, fit_normaliser, restore_cache
from adjointrwm.eval import inner_validation_split, kfold_assignment

V1_DIR = DRIVE_ROOT / 'runs' / CFG.v1_run_id
V2_DIR = DRIVE_ROOT / 'runs' / CFG.v2_run_id

CACHE_MANIFEST_PATH = V2_DIR / 'cache' / 'cache_manifest.json'
cache_manifest = json.loads(CACHE_MANIFEST_PATH.read_text())
records = restore_cache(cache_manifest, LOCAL / 'cache')
BY_ID = {r['episode_id']: r for r in records}
EPISODE_IDS = sorted(BY_ID)
assert len(EPISODE_IDS) == len(records) == 100, f'expected the 100 DROID-100 episodes, found {len(records)}'
print(f'Restored {len(records)} cached episodes (hashes verified).')

FOLDS = kfold_assignment(EPISODE_IDS, CFG.k)
assert {f: list(FOLDS.values()).count(f) for f in range(CFG.k)} == {f: len(EPISODE_IDS) // CFG.k for f in range(CFG.k)}
PILOT_SPLIT = episode_split(EPISODE_IDS)
assert PILOT_SPLIT == assignment_from_manifest(json.loads((V1_DIR / 'config' / 'data_manifest.json').read_text())), 'the episode split differs from the pilot'

SPEC = WindowSpec(CFG.context_len, CFG.horizon, CFG.window_stride)
DATA_MANIFEST = {'episodes': EPISODE_IDS, 'fold_of_episode': FOLDS, 'pilot_split': PILOT_SPLIT, 'cache_manifest_sha256': sha256_file(CACHE_MANIFEST_PATH)}
DATA_MANIFEST_HASH = sha256_json(DATA_MANIFEST)
SOURCE_HASH = sha256_json({'repo_commit': REPO_COMMIT, 'repo_dirty': REPO_DIRTY})
atomic_write_json(PATHS['config'] / 'data_manifest.json', {**DATA_MANIFEST, 'data_manifest_sha256': DATA_MANIFEST_HASH})


def load_arrays(record):
    with np.load(record['cached_path']) as episode:
        return {k: episode[k] for k in ('states', 'actions')}


def make_datasets(train_ids, validation_ids, test_ids):
    """Datasets for one unit. The input normaliser is fitted on the training episodes only."""
    assert not (set(train_ids) & set(validation_ids)) and not (set(train_ids) & set(test_ids)) and not (set(validation_ids) & set(test_ids))
    arrays = [load_arrays(BY_ID[i]) for i in train_ids]
    normaliser = fit_normaliser([a['states'] for a in arrays], [a['actions'] for a in arrays])
    build = lambda ids: WindowDataset([BY_ID[i] for i in sorted(ids)], SPEC, normaliser, visual_layout='flat') if ids else None  # noqa: E731
    return build(train_ids), build(validation_ids), build(test_ids)


def unit_plan(unit):
    """``(train ids, validation ids, held-out ids, seed)`` of a unit: ``'anchor'`` is the pilot's split (its test episodes stay untouched), an integer is a held-out fold."""
    if unit == 'anchor':
        return ([e for e in EPISODE_IDS if PILOT_SPLIT[e] == 'train'], [e for e in EPISODE_IDS if PILOT_SPLIT[e] == 'validation'], [], CFG.seed_base)
    held_out = [e for e in EPISODE_IDS if FOLDS[e] == unit]
    train, validation = inner_validation_split([e for e in EPISODE_IDS if FOLDS[e] != unit], CFG.inner_validation_episodes)
    return train, validation, held_out, CFG.seed_base + 1 + unit


for unit in ['anchor', *range(CFG.k)]:
    train_ids, validation_ids, held_out_ids, _ = unit_plan(unit)
    print(f'unit {unit!s:>6}: train {len(train_ids):3d} | validation {len(validation_ids):3d} | held out {len(held_out_ids):3d} episodes')
sample_train = make_datasets(*unit_plan('anchor')[:3])[0]
sample = sample_train[0]
STATE_DIM, ACTION_DIM, VISUAL_DIM = sample['context_state'].shape[-1], sample['context_action'].shape[-1], sample['context_visual'].shape[-1]
def stored_anchor(regime):
    """The number the anchor unit must reproduce, and where it comes from. Read (and the pilot checkpoint hash-checked) before any training, so a wrong file fails in seconds."""
    if regime == 'v2':
        stored = json.loads((V2_DIR / 'artifacts' / 'dynamics_gate.json').read_text())['0']
        return float(stored['model_rmse']), "probe run dynamics_gate.json, seed 0, validation, mode base", CFG.anchor_atol_v2
    path = V1_DIR / 'checkpoints' / 'best_dynamics.pt'
    actual = sha256_file(path)
    if actual != CFG.v1_checkpoint_sha256:
        raise RuntimeError(f'{path} has SHA-256 {actual}, expected {CFG.v1_checkpoint_sha256}.')
    payload = torch.load(path, map_location='cpu', weights_only=False)   # our own file, hash-checked just above
    return float(np.mean(payload['best_metrics']['full_rmse_by_horizon'])), "pilot checkpoint best_metrics, validation, mean of full_rmse_by_horizon", CFG.anchor_atol_pilot



ANCHORS_STORED = {regime: stored_anchor(regime) for regime in CFG.regimes}
print('stored anchors:', {r: (round(v[0], 6), v[2]) for r, v in ANCHORS_STORED.items()})
print('dims:', STATE_DIM, ACTION_DIM, VISUAL_DIM)

In [ ]:
# ================================================================
# 3. Train and score every unit of every requested regime (anchor unit first, then the five folds)
# ================================================================
import time

from adjointrwm.eval import episode_error_table, pooled_relative_improvement, stack_windows
from adjointrwm.models import AdjointRecursiveWorldModel, AdjointRWMConfig
from adjointrwm.training import TrainConfig, load_best_weights, predict_dataset, seed_everything, train_job


def build_model(regime):
    spec = REGIMES[regime]
    config = AdjointRWMConfig(context_len=CFG.context_len, horizon=CFG.horizon, d_model=CFG.d_model, transformer_layers=CFG.transformer_layers,
                              transformer_heads=CFG.transformer_heads, transformer_ff=CFG.transformer_ff, dropout=CFG.dropout,
                              num_refinement_candidates=CFG.num_refinement_candidates, candidate_costs=CFG.candidate_costs, rate_beta=CFG.rate_beta,
                              mask_mode=spec['mask_mode'], prediction_mode=spec['prediction_mode'])
    return AdjointRecursiveWorldModel(STATE_DIM, ACTION_DIM, VISUAL_DIM, config)


def train_config(seed):
    return TrainConfig(steps=CFG.steps, batch_size=CFG.batch_size, lr=CFG.learning_rate, weight_decay=CFG.weight_decay, warmup_steps=CFG.warmup_steps,
                       grad_clip=CFG.gradient_clip, eval_interval=CFG.eval_interval, checkpoint_interval=CFG.checkpoint_interval, seed=seed, amp=True,
                       eval_batch_size=CFG.eval_batch_size)


def score_heldout(model, dataset, mode):
    """Per-episode summed squared errors of the model and of persistence on a held-out dataset, in the given prediction mode."""
    model.config = dataclasses.replace(model.config, prediction_mode=mode)
    pred = predict_dataset(model, dataset, CFG.eval_batch_size, DEVICE)
    context = stack_windows(dataset, keys=['context_state'])['context_state']
    return episode_error_table(pred['state_mean'], pred['target_state'], context, pred['episode_id'], CFG.horizon)


def run_unit(regime, unit):
    spec = REGIMES[regime]
    train_ids, validation_ids, held_out_ids, seed = unit_plan(unit)
    modes = (spec['prediction_mode'], 'full' if spec['prediction_mode'] == 'base' else 'base')          # the regime's own gate mode first
    train_ds, validation_ds, held_ds = make_datasets(train_ids, validation_ids, held_out_ids)
    identity = {'run_id': RUN_ID, 'arm': f'kfold_{regime}_{unit}', 'seed': int(seed), 'config_hash': CONFIG_HASH, 'data_manifest_hash': DATA_MANIFEST_HASH, 'source_hash': SOURCE_HASH}
    unit_dir = LOCAL / 'units' / f'{regime}_{unit}'
    seed_everything(seed)
    model = build_model(regime)
    began = time.time()
    summary = train_job(model, train_ds, validation_ds, train_config(seed), identity=identity, local_dir=unit_dir / 'local', persist_dir=unit_dir / 'persist',
                        device=DEVICE, stage='dynamics')
    if summary['status'] != 'DONE':
        raise RuntimeError(f'{regime} unit {unit}: training ended with status {summary["status"]}')
    result = {'regime': regime, 'unit': unit, 'seed': int(seed), 'modes': list(modes), 'train_episodes': len(train_ids), 'validation_episodes': len(validation_ids),
              'held_out_episodes': len(held_out_ids), 'global_step': summary['global_step'], 'best': summary['best'], 'session_seconds': summary['session_seconds'],
              'peak_vram_gib': summary['peak_vram_gib'], 'best_checkpoint_sha256': summary['best']['checkpoint']['sha256'], 'tables': {}}
    if held_ds is not None:
        for kind in ('final', 'best'):           # the in-memory model still holds the final-step weights; best.pt is loaded second
            if kind == 'best':
                load_best_weights(model, unit_dir / 'persist', identity=identity)
            model.eval()
            for mode in modes:
                result['tables'][f'{kind}|{mode}'] = score_heldout(model, held_ds, mode).to_dict('records')
    log_path = unit_dir / 'local' / 'train_log.jsonl'
    if log_path.exists():
        atomic_copy(log_path, PATHS['logs'] / f'train_log_{regime}_{unit}.jsonl')
    result['wall_seconds'] = time.time() - began
    atomic_write_json(UNIT_DIR / f'{regime}_{unit}.json', result)
    del model
    shutil.rmtree(unit_dir, ignore_errors=True)          # checkpoints stay on the VM disk only until here; their SHA-256 is in the result
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return result


for regime in CFG.regimes:
    for unit in ['anchor', *range(CFG.k)]:
        if (UNIT_DIR / f'{regime}_{unit}.json').exists():
            print(f'{regime} unit {unit}: already done (resume)')
            continue
        result = run_unit(regime, unit)
        print(f"{regime} unit {unit!s:>6}: best validation {result['best']['validation_score']:.4f} at step {result['best']['step']} | {result['wall_seconds']:.0f} s")

In [ ]:
# ================================================================
# 4. Anchors, the pooled estimate per regime, the frozen classification, report
# ================================================================
from adjointrwm.eval import (
    anchor_check, classify_against_margin, cluster_bootstrap_relative_improvement, random_subset_gate_rate,
)


def load_units(regime):
    return {unit: json.loads((UNIT_DIR / f'{regime}_{unit}.json').read_text()) for unit in ['anchor', *range(CFG.k)]}


def pooled_tables(units, regime):
    """``{(kind, mode): episode table over all 100 held-out episodes}`` with a ``fold`` column; every episode must appear exactly once."""
    out = {}
    for key in units[0]['tables']:
        frames = [pd.DataFrame(units[f]['tables'][key]).assign(fold=f) for f in range(CFG.k)]
        table = pd.concat(frames, ignore_index=True)
        assert sorted(table['episode_id']) == EPISODE_IDS, f'{regime} {key}: the held-out episodes are not exactly the 100 episodes'
        kind, mode = key.split('|')
        out[(kind, mode)] = table
    return out


REPORT = {'run_id': RUN_ID, 'config_hash': CONFIG_HASH, 'repo_commit': REPO_COMMIT, 'plan': 'docs/plans/dynamics-kfold-plan.md', 'kfold_uses_all_100_episodes': True,
          'anchor_reads_pilot_test_episodes': False, 'regimes': {}}
lines = [f'# Dynamics k-fold `{RUN_ID}`', '']
all_csv = []
for regime in CFG.regimes:
    units = load_units(regime)
    stored, source, atol = ANCHORS_STORED[regime]
    anchor = anchor_check(units['anchor']['best']['validation_score'], stored, atol)
    anchor.update({'source': source, 'best_step': units['anchor']['best']['step']})
    own = REGIMES[regime]['prediction_mode']
    tables = pooled_tables(units, regime)
    entry = {'anchor': anchor, 'own_mode': own, 'units': {str(u): {k: v for k, v in r.items() if k != 'tables'} for u, r in units.items()}, 'estimates': {}}
    for (kind, mode), table in tables.items():
        boot = cluster_bootstrap_relative_improvement(table, CFG.bootstrap_resamples, CFG.bootstrap_seed)
        pooled = pooled_relative_improvement(table)
        per_fold = [pooled_relative_improvement(table[table['fold'] == f])['relative_improvement'] for f in range(CFG.k)]
        per_episode = (np.sqrt(table[[f'model_sse_h{h}' for h in range(1, CFG.horizon + 1)]].div(table['windows'], axis=0)).mean(axis=1)
                       < np.sqrt(table[[f'persistence_sse_h{h}' for h in range(1, CFG.horizon + 1)]].div(table['windows'], axis=0)).mean(axis=1))
        entry['estimates'][f'{kind}|{mode}'] = {
            'kind': kind, 'mode': mode, 'primary': (kind == 'best' and mode == own), 'pooled': pooled, 'bootstrap': boot, 'per_fold_relative_improvement': per_fold,
            'fraction_of_episodes_model_beats_persistence': float(per_episode.mean()),
            'random_ten_episode_split_gate_rate': random_subset_gate_rate(table, CFG.subset_size, CFG.subset_draws, CFG.bootstrap_seed, CFG.margin),
            'class': classify_against_margin(boot['ci_low'], boot['ci_high'], CFG.margin)}
        all_csv.append(table.assign(regime=regime, kind=kind, mode=mode))
    primary = entry['estimates'][f'best|{own}']
    entry['status'] = 'OK' if anchor['within'] else 'ANCHOR_FAILED'
    entry['primary_class'] = primary['class'] if anchor['within'] else None
    REPORT['regimes'][regime] = entry
    lines += [f'## Regime `{regime}` (own mode `{own}`): **{entry["status"]}**' + (f', class **{entry["primary_class"]}**' if entry['primary_class'] else ', no classification (anchor outside its tolerance)'), '',
              f"Anchor: recomputed best validation {anchor['recomputed']:.4f} against stored {anchor['stored']:.4f} ({source}), difference {anchor['abs_diff']:.4f}, tolerance {anchor['atol']}, within: {anchor['within']}.", '',
              '| Checkpoint | Mode | Relative improvement R | 95 % interval | Class | Random 10-episode splits passing 2 % | Episodes where the model beats persistence |', '|---|---|---:|---|---|---:|---:|']
    for key, e in entry['estimates'].items():
        b = e['bootstrap']
        lines.append(f"| {e['kind']} | {e['mode']}{' (primary)' if e['primary'] else ''} | {b['estimate']:+.3f} | [{b['ci_low']:+.3f}, {b['ci_high']:+.3f}] | {e['class']} | "
                     f"{e['random_ten_episode_split_gate_rate']['pass_rate']:.3f} | {e['fraction_of_episodes_model_beats_persistence']:.2f} |")
    lines += ['', 'Per fold (primary): ' + ', '.join(f'{x:+.3f}' for x in primary['per_fold_relative_improvement']),
              'By horizon step (primary, estimate [95 % interval]): ' + '; '.join(f"step {h['step']}: {h['estimate']:+.3f} [{h['ci_low']:+.3f}, {h['ci_high']:+.3f}]" for h in primary['bootstrap']['by_horizon']), '']
pd.concat(all_csv, ignore_index=True).to_csv(PATHS['artifacts'] / 'episode_errors.csv', index=False)
REPORT['status'] = 'OK' if all(e['status'] == 'OK' for e in REPORT['regimes'].values()) else 'ANCHOR_FAILED'
REPORT['claim_boundary'] = ('Episode-level k-fold dynamics against persistence on DROID-100, one training seed per fold, 70 training episodes per fold. '
                            'Diagnostic models; not evidence for or against H2; B2 does not reuse them.')
atomic_write_json(PATHS['reports'] / 'acceptance_report.json', REPORT)
atomic_write_text(PATHS['reports'] / 'run_summary.md', '\n'.join([*lines, f"Status: **{REPORT['status']}**.", '', REPORT['claim_boundary']]) + '\n')
if REPORT['status'] == 'OK':
    atomic_write_text(RUN_DIR / 'COMPLETE', datetime.datetime.now(datetime.UTC).isoformat() + '\n')
print('\n'.join(lines))
print('Status:', REPORT['status'], '| Run directory:', RUN_DIR)